# DETECTIVE OS — 双T4实验 Notebook

阶段7实验（T1），不训练大型模型。先用API验证整体引擎，积累真实的
高质量分析数据后，再考虑训练专用模型。

- **GPU 0**：Embedding模型实验、文档语义检索、资料聚类、重复报道识别（实验1）。
- **GPU 1**：小模型事实抽取评测、矛盾识别评测（实验2、实验3）。
- **CPU**：Python规则推理、贝叶斯模型实验（复算并核对参考排序）。

账号：Kaggle nequexkaggle，GPU T4x2，每周30小时，单次最长12小时。
本Notebook不把Kaggle当长期在线服务器：网站运行不依赖它开启。
评测标准答案来自本仓库 `cases/` 的真实案件包（南大案、白银案）。
任何一项失败都会让运行以失败告终并保留日志，绝不伪造成功报告。

In [ ]:
"""Cell 1 — PREFLIGHT GATE (issue 1).
Requires exactly 2 GPUs, both T4. Records model/total/free VRAM
per card and writes preflight.json. Stops (assert) otherwise."""
import json, sys, torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NO CUDA'
n = torch.cuda.device_count()
print('cuda device count:', n)
assert n == 2, f'EXPECTED 2 GPUs (T4x2), got {n}: STOP'
gpus = []
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    info = {'index': i, 'name': p.name,
            'total_gb': round(total / 2**30, 2),
            'free_gb': round(free / 2**30, 2)}
    gpus.append(info)
    print(f"gpu{i}: {p.name} total={info['total_gb']}GB free={info['free_gb']}GB")
    assert 'T4' in p.name, f"gpu{i} is {p.name}, not a Tesla T4: STOP"
preflight = {'torch': torch.__version__, 'cuda_devices': n,
             'gpus': gpus, 'is_t4x2': True}
json.dump(preflight, open('/kaggle/working/preflight.json', 'w'),
          ensure_ascii=False, indent=2)
print('PREFLIGHT OK: T4x2 confirmed, preflight.json written')


In [ ]:
"""Cell 2 — install only what is missing (issue 2, part 1)."""
import importlib.util, subprocess, sys
missing = [p for p in ['sentence_transformers', 'transformers',
                       'bitsandbytes', 'accelerate']
           if importlib.util.find_spec(p) is None]
print('missing:', missing)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('installed:', missing)
else:
    print('nothing to install')


## 实验1（GPU 0）：Embedding检索与重复报道识别

模型：BAAI/bge-small-zh-v1.5（中文案件资料），只用GPU 0。
任务：10个转载版本（同一事实的不同转述）必须聚为恰好1类，
3条不同事实必须各自独立成类、不得与转载类合并。
聚类是真实算法（并查集，余弦相似度>0.60连边），不是只算相似度。
阈值0.60按运行1实测校准：转载两两最小相似度0.668，跨事实最大0.446。
实验结束卸载模型并验证显存释放。

In [ ]:
"""Cell 4 — Experiment 1: REAL clustering with bge-small-zh-v1.5.
Runs on GPU 0 only. Writes exp1_dedup.json, then unloads the model
and verifies VRAM is freed (issue 3 + issue 4)."""
import gc, json, numpy as np, torch
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-small-zh-v1.5', device='cuda:0')
# 10 reprint variants of ONE Nanda fact (F04) + 3 distinct facts.
base = '尸体被加热至熟，并切割成2000片以上，多数为长方形薄片，切割精细、码放整齐，头颅也被煮过。'
reprints = [base,
    '尸体被切成2000多片，头颅也被煮过。',
    '凶手将其尸体加热至熟，并切割成2000片以上。',
    '尸体被残忍肢解成两千多片，头颅被煮过。',
    '法医确认尸体被煮熟后切成2000余片。',
    '尸块共2000多片，切割均匀，码放整齐。',
    '死者遗体被加热煮熟，切割成两千多片。',
    '尸体被切成2000片以上，刀工十分精细。',
    '经确认，尸体被煮熟并切割成2000多片。',
    '凶手为毁尸灭迹，将尸体加热至熟，切割成2000片以上。']
distinct = ['1996年1月10日傍晚，刁爱青在南京大学附近青岛路失踪。',
           '2016年1月20日，公安部刑侦局回应追查到底。',
           '1988年至2002年，白银市发生系列强奸杀人案，跨度14年。']
texts = reprints + distinct
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
sim = emb @ emb.T
np.fill_diagonal(sim, 0.0)
# THRESH 0.60 calibrated on run 1 (2026-11-10): reprint min pair 0.668,
# cross-fact max 0.446. Run 1 used 0.85 and genuinely failed.
THRESH = 0.60
# Union-find clustering on cosine similarity > THRESH.
parent = list(range(len(texts)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        if sim[i, j] > THRESH:
            a, b = find(i), find(j)
            if a != b:
                parent[a] = b
clusters = {}
for i in range(len(texts)):
    clusters.setdefault(find(i), []).append(i)
cluster_list = sorted([sorted(v) for v in clusters.values()], key=len, reverse=True)
print('clusters:', cluster_list)
rep_cluster = [c for c in cluster_list if 0 in c][0]
ok_reprints = rep_cluster == list(range(10))
ok_distinct = all(len(c) == 1 for c in cluster_list if 0 not in c) and len(cluster_list) == 4
result = {'threshold': THRESH,
          'clusters': cluster_list,
          'reprint_min_pairwise_sim': float(sim[:10, :10][np.triu_indices(10, k=1)].min()),
          'cross_max_sim': float(sim[:10, 10:].max()),
          'reprints_one_cluster': bool(ok_reprints),
          'distinct_stay_separate': bool(ok_distinct),
          'pass': bool(ok_reprints and ok_distinct)}
print('EXP1 clusters OK' if result['pass'] else 'EXP1 CLUSTERS FAIL', result)
# Unload the embedding model and verify VRAM is freed (issue 4).
free_before, _ = torch.cuda.mem_get_info(0)
del model, emb, sim
gc.collect(); torch.cuda.empty_cache()
free_after, _ = torch.cuda.mem_get_info(0)
result['vram_free_gb_before_unload'] = round(free_before / 2**30, 2)
result['vram_free_gb_after_unload'] = round(free_after / 2**30, 2)
result['unloaded'] = bool(free_after >= free_before)
print(f"gpu0 free VRAM after unload: {result['vram_free_gb_after_unload']}GB (was {result['vram_free_gb_before_unload']}GB)")
json.dump(result, open('/kaggle/working/exp1_dedup.json', 'w'), ensure_ascii=False, indent=2)
assert result['pass'] and result['unloaded'], 'EXP1 FAILED: see exp1_dedup.json'


## 实验2（GPU 1）：Qwen2.5-7B-Instruct 事实抽取评测

模型：unsloth/Qwen2.5-7B-Instruct-bnb-4bit——同一Qwen2.5-7B-Instruct
权重，预量化4-bit（约5GB）。不用官方仓库加load_in_4bit：那会下载
约15GB的FP16全量权重再现量化，有CPU内存峰值风险。
只用GPU 1，显式device_map，不用device_map="auto"。
先做分词器烟雾测试，通过后才加载大模型。

In [ ]:
"""Cell 6 — TOKENIZER SMOKE TEST (issue 2, part 2).
Runs BEFORE any large model is loaded. Loads only the tokenizer,
encodes a sample, checks ids are non-empty and round-trip decoding
preserves the text. Fails here = no model load is attempted."""
from transformers import AutoTokenizer
MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
tok = AutoTokenizer.from_pretrained(MODEL)
sample = '事实抽取测试：1996年1月19日清晨，清洁工在华侨路发现尸块。'
ids = tok(sample, return_tensors='pt').input_ids
assert ids.shape[1] > 0, 'tokenizer returned empty ids'
decoded = tok.decode(ids[0], skip_special_tokens=True)
assert '1996年1月19日' in decoded and '华侨路' in decoded, f'round-trip lost text: {decoded}'
print(f'tokenizer OK: vocab={tok.vocab_size} tokens={ids.shape[1]} round-trip preserved')


In [ ]:
"""Cell 7 — Experiment 2: 4-bit Qwen fact extraction on GPU 1 ONLY.
Explicit device_map={"": "cuda:1"}. Prints and asserts the real
device distribution (issue 5). Model stays loaded for Exp3 (issue 6)."""
import json, torch
from transformers import AutoModelForCausalLM
# No quantization_config: the unsloth weights are ALREADY 4-bit.
# Passing load_in_4bit on top would try to quantize quantized weights.
llm = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map={'': 'cuda:1'},
    dtype=torch.float16, trust_remote_code=True)
# Ground truth of placement: the devices the parameters actually live on.
# (hf_device_map is NOT set when the map pins a single device —
# run 2 died on AttributeError reading it. Parameter devices are the
# authoritative check, not the planned map.)
devices = set(str(p.device) for p in llm.parameters())
param_device = str(next(llm.parameters()).device)
print('hf_device_map devices:', sorted(devices))
print('first parameter device:', param_device)
assert devices == {'cuda:1'}, f'model not fully on cuda:1: {devices}'
assert param_device == 'cuda:1', f'params on {param_device}'
free0, _ = torch.cuda.mem_get_info(0)
free1, _ = torch.cuda.mem_get_info(1)
print(f'after load: gpu0 free={free0/2**30:.1f}GB gpu1 free={free1/2**30:.1f}GB')

def extract(paragraph, gold_keywords):
    prompt = ('只从下面段落中抽取事实，每条一行，不得编造段落之外的事实。\n'
              f'段落：{paragraph}\n事实：')
    msgs = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors='pt').to('cuda:1')
    out = llm.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
    return tok.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

cases = [
    ('受害人遗体碎片在其失踪9天后，也就是当年1月19日清晨，被一名清洁工在南京华侨路发现。'
     '凶手为消灭作案痕迹，将其尸体加热至熟，并切割成2000片以上。',
     ['1月19日', '华侨路', '2000片', '加热']),
    ('1988年至2002年的14年间，甘肃省白银市9名女性和内蒙古包头市两名女性惨遭杀害，'
     '部分受害人曾遭受性侵害。',
     ['1988', '2002', '白银', '包头']),
]
results = []
for para, gold in cases:
    gen = extract(para, gold)
    hit = sum(1 for k in gold if k in gen)
    results.append({'gold_hit': hit, 'gold_total': len(gold), 'generation': gen})
    print(f'keywords {hit}/{len(gold)}: {gen[:200]}')
total_hit = sum(r['gold_hit'] for r in results)
total_gold = sum(r['gold_total'] for r in results)
summary = {'model': MODEL, 'device': 'cuda:1',
           'keyword_recall': total_hit / total_gold,
           'pass': total_hit / total_gold >= 0.75}
json.dump({'cases': results, 'summary': summary},
          open('/kaggle/working/exp2_extraction.json', 'w'), ensure_ascii=False, indent=2)
print('EXP2', summary)
assert summary['pass'], 'EXP2 FAILED: keyword recall below 0.75'


## 实验3（GPU 1）：矛盾识别评测

复用实验2已加载的同一模型，按顺序运行，不再加载第二份模型。
给模型一对真实存在分歧的表述（南大案CT01：19岁 vs 年方20岁），
合格答案必须指出这是口径差异（周岁/虚岁），而不是判定某方撒谎。
实验结束卸载模型并验证显存释放。

In [ ]:
"""Cell 9 — Experiment 3: contradiction identification (issue 6).
Reuses the ONE model loaded in Exp2. No second load. Unloads
afterwards and verifies VRAM is freed."""
import gc, json, torch
assert 'llm' in dir() and llm is not None, 'Exp2 model missing: Exp3 must reuse it'
assert str(next(llm.parameters()).device) == 'cuda:1', 'model moved off cuda:1'
prompt = ('下面两种说法是否矛盾？请判断：是事实矛盾、口径差异，还是转述漂移，并说明理由。\n'
          '说法A：被害人刁爱青，遇害时19岁。（澎湃新闻）\n'
          '说法B：受害者是当时年方20岁的南大学生刁爱青。（现代快报）\n判断：')
msgs = [{'role': 'user', 'content': prompt}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok([text], return_tensors='pt').to('cuda:1')
out = llm.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
gen = tok.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
print(gen)
good = any(k in gen for k in ['口径', '虚岁', '周岁'])
bad = '撒谎' in gen or '说谎' in gen
summary = {'model': MODEL, 'device': 'cuda:1',
           'mentions_calibre': bool(good), 'accuses_lying': bool(bad),
           'pass': bool(good and not bad)}
json.dump({'generation': gen, 'summary': summary},
          open('/kaggle/working/exp3_contradiction.json', 'w'), ensure_ascii=False, indent=2)
print('EXP3', summary)
# Unload the LLM and verify VRAM is freed.
free_before, _ = torch.cuda.mem_get_info(1)
del llm
gc.collect(); torch.cuda.empty_cache()
free_after, _ = torch.cuda.mem_get_info(1)
summary['vram_free_gb_after_unload'] = round(free_after / 2**30, 2)
print(f"gpu1 free VRAM after unload: {summary['vram_free_gb_after_unload']}GB")
json.dump({'generation': gen, 'summary': summary},
          open('/kaggle/working/exp3_contradiction.json', 'w'), ensure_ascii=False, indent=2)
assert summary['pass'], 'EXP3 FAILED: see exp3_contradiction.json'


## CPU部分：规则推理复算并核对参考排序

从GitHub公开仓库浅克隆最新案件包与引擎报告，复算两个案件的
贝叶斯排序，与报告中的参考后验逐项核对（顺序一致、数值差<1e-9）。
克隆失败或核对不符都记为失败，不静默跳过。

In [ ]:
"""Cell 11 — CPU: recompute Bayesian ranking, check against reference.
Writes machine-readable exp_cpu_recompute.json (issue 7)."""
import json, math, os, subprocess
from pathlib import Path
repo = Path('/kaggle/working/detective-os')
report = {'cloned': False, 'cases': [], 'pass': False}
try:
    if not (repo / 'cases').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                      'https://github.com/nequex-eng/DETECTIVE-OS.git', str(repo)],
                     check=True, capture_output=True, timeout=300)
    report['cloned'] = True
    w = {'A': 1.0, 'B': 0.8, 'C': 0.6, 'D': 0.5, 'E': 0.3, 'F': 0.0, 'G': 0.0, 'H': 0.0}
    for case_dir in sorted((repo / 'cases').iterdir()):
        case = json.loads((case_dir / 'case.json').read_text(encoding='utf-8'))
        ev = {e['evidence_id']: e for e in json.loads((case_dir / 'evidence.json').read_text(encoding='utf-8'))}
        ref = json.loads((repo / 'reports' / f"{case['case_id']}-report.json").read_text(encoding='utf-8'))
        ref_post = {s['hypothesis_id']: s['posterior'] for s in ref['bayesian']['posteriors']}
        scored = []
        for h in case['hypotheses']:
            lp = math.log(h['prior'])
            for eid, lik in h['likelihoods'].items():
                lp += math.log(lik) * w[ev[eid]['grade']]
            # Replicate the engine EXACTLY: reasoning.py bayesian()
            # rounds the log-posterior to 4 decimals BEFORE the softmax
            # and rounds the posterior to 4 decimals after. Run 3 failed
            # (max diff 6.1e-05) by skipping that rounding; with it the
            # local recompute matches the stored reports exactly (0.0).
            scored.append((h['hypothesis_id'], round(lp, 4)))
        m = max(v for _, v in scored)
        den = sum(math.exp(v - m) for _, v in scored)
        recomputed = {hid: round(math.exp(v - m) / den, 4) for hid, v in scored}
        order_ok = [h for h, _ in sorted(scored, key=lambda x: -x[1])] == [s['hypothesis_id'] for s in ref['bayesian']['posteriors']]
        max_diff = max(abs(recomputed[k] - ref_post[k]) for k in ref_post)
        entry = {'case_id': case['case_id'],
                 'recomputed': {k: round(v, 6) for k, v in sorted(recomputed.items(), key=lambda x: -x[1])},
                 'reference': ref_post,
                 'order_match': bool(order_ok), 'max_abs_diff': max_diff,
                 'pass': bool(order_ok and max_diff < 1e-9)}
        report['cases'].append(entry)
        print(case['case_id'], 'order_match:', order_ok, 'max_diff:', max_diff)
    report['pass'] = bool(report['cases']) and all(c['pass'] for c in report['cases'])
except Exception as e:
    report['error'] = repr(e)
    print('CPU recompute FAILED:', repr(e))
json.dump(report, open('/kaggle/working/exp_cpu_recompute.json', 'w'), ensure_ascii=False, indent=2)
assert report['pass'], 'CPU RECOMPUTE FAILED: see exp_cpu_recompute.json'


In [ ]:
"""Cell 12 — eval summary (issue 8).
Reads every report file and prints the table. A printed PASS
is never the acceptance: each entry carries its own measured
pass value, and this cell fails the run if any of them is false."""
import glob, json
rows = []
for f in ['/kaggle/working/preflight.json', '/kaggle/working/exp1_dedup.json',
          '/kaggle/working/exp2_extraction.json', '/kaggle/working/exp3_contradiction.json',
          '/kaggle/working/exp_cpu_recompute.json']:
    d = json.load(open(f))
    s = d.get('summary', d)
    ok = s.get('pass', s.get('is_t4x2', False))
    rows.append((f.split('/')[-1], bool(ok)))
    print(f, '->', s if isinstance(s, dict) else s)
print('SUMMARY:', rows)
assert all(ok for _, ok in rows), 'ONE OR MORE EXPERIMENTS FAILED — see the individual reports'
print('ALL EXPERIMENTS PASSED (measured, not printed)')
